# Task 4 — Forecasting Access and Usage (2025–2027)

Objective: produce baseline and event-augmented forecasts for Account Ownership (`ACC_OWNERSHIP`) and Digital Payment Usage (estimated from `ACC_MM_ACCOUNT` × `USG_ACTIVE_RATE`), with scenarios and confidence intervals.

This notebook contains:
- Data loading and preprocessing
- Trend models (linear) with confidence intervals
- Event-augmented forecasts using documented event effects
- Scenario analysis (pessimistic / base / optimistic)
- Forecast tables and visualizations saved to `reports/analysis_outputs/`

Run order: execute cells sequentially.

In [ ]:
# Imports and paths
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings

ROOT = Path('..')
DATA_PROCESSED = ROOT / 'data' / 'processed'
DATA_RAW = ROOT / 'data' / 'raw'
OUT = Path('reports') / 'analysis_outputs'
OUT.mkdir(parents=True, exist_ok=True)

pd.set_option('display.width', 200)
warnings.filterwarnings('ignore')

In [ ]:
# Load enriched dataset and impact links
enriched_fp = DATA_PROCESSED / 'ethiopia_fi_enriched_data.csv'
impact_fp = DATA_RAW / 'Impact_sheet.csv'
enriched = pd.read_csv(enriched_fp)
impact = pd.read_csv(impact_fp)

# Quick preview
display(enriched.head())
display(impact.head())

## Helper functions: annual series and event effects

In [ ]:
def annual_observed(enriched, indicator_code):
    df = enriched[enriched['indicator_code'] == indicator_code].copy()
    df = df[df['record_type'] == 'observation']
    df['observation_date'] = pd.to_datetime(df['observation_date'], errors='coerce')
    df['year'] = df['observation_date'].dt.year.fillna(df['fiscal_year']).astype('Int64')
    s = df.groupby('year')['value_numeric'].mean().sort_index()
    s.index = s.index.astype(int)
    return s

# Build digital payment usage proxy = percent adults with account * activity rate
def digital_usage_series(enriched):
    acc = annual_observed(enriched, 'ACC_MM_ACCOUNT')
    act = annual_observed(enriched, 'USG_ACTIVE_RATE')
    # Align years; activity rate in percent -> fraction
    years = sorted(set(acc.index) | set(act.index))
    out = pd.Series(index=years, dtype=float)
    for y in years:
        a = acc.get(y, np.nan)
        r = act.get(y, np.nan)
        if pd.isna(a) and pd.isna(r):
            out.loc[y] = np.nan
        else:
            out.loc[y] = (0 if pd.isna(a) else a) * (0 if pd.isna(r) else r) / 100.0
    return out.sort_index()

In [ ]:
# Prepare observed series
obs_acc = annual_observed(enriched, 'ACC_OWNERSHIP')
obs_usage = digital_usage_series(enriched)

print('Account Ownership (observed):')
display(obs_acc)
print('Digital Payment Usage (proxy) — ACC_MM_ACCOUNT * USG_ACTIVE_RATE:')
display(obs_usage)


## Trend model (linear) with confidence intervals — helper using statsmodels if available

In [ ]:
def fit_linear_ci(series):
    # Fit year -> value, return fitted function and 95% CI for predictions
    import statsmodels.api as sm
    df = series.dropna().reset_index()
    df.columns = ['year', 'y']
    X = sm.add_constant(df['year'])
    model = sm.OLS(df['y'], X).fit()
    return model

def predict_with_ci(model, years):
    import statsmodels.api as sm
    Xnew = sm.add_constant(pd.Series(years))
    pred = model.get_prediction(Xnew)
    summary = pred.summary_frame(alpha=0.05)
    return summary[['mean','mean_ci_lower','mean_ci_upper']].set_index(pd.Index(years))
: 
,
: {
: 

: [
,
2025
2026
2027
,
,
,
,
,
,
,
,
,
,

: 
,
: {
: 

: [
,

: 
,
: {
: 

: [
,
,
,
,
,
,
,
-1
,
,
,
,
,